# Lab 3 - FSRCNN Residual 96/40/m8 Resume 2750 (Colab)

- Model ID: `fsrcnn_residual_96_40_m8`
- Goal: resume fixed-resolution super-resolution/restoration training from the 085328_2604_fsrcnn_residual_96_40_m8 last.pth checkpoint at epoch 1138
- Expected input/output: `256x256x3`
- Runtime target: Google Colab with Google Drive
- Dataset layout: TinySRCNN legacy Lab 3 Drive folders
- NPU-facing operators: `Conv`, `LeakyRelu`, and final residual `Add`

## 1. Setup

In [ ]:
try:
    import google.colab  # type: ignore
    IN_COLAB_BOOTSTRAP = True
except Exception:
    IN_COLAB_BOOTSTRAP = False

if IN_COLAB_BOOTSTRAP:
    %pip install -q onnx onnxruntime

import csv
import json
import math
import os
import random
import sys
import time
from contextlib import nullcontext
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from typing import Any

import numpy as np
import onnx
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset

IMAGE_SUFFIXES = {'.png', '.jpg', '.jpeg', '.bmp'}
DRIVE_ROOT = Path('/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3')
DRIVE_DATA_ROOT = DRIVE_ROOT / 'Data'
DRIVE_RUNS_ROOT = DRIVE_ROOT / 'runs'

try:
    BICUBIC_RESAMPLE = Image.Resampling.BICUBIC
except AttributeError:
    BICUBIC_RESAMPLE = Image.BICUBIC

try:
    FLIP_LEFT_RIGHT = Image.Transpose.FLIP_LEFT_RIGHT
    FLIP_TOP_BOTTOM = Image.Transpose.FLIP_TOP_BOTTOM
    ROTATE_90 = Image.Transpose.ROTATE_90
    ROTATE_180 = Image.Transpose.ROTATE_180
    ROTATE_270 = Image.Transpose.ROTATE_270
except AttributeError:
    FLIP_LEFT_RIGHT = Image.FLIP_LEFT_RIGHT
    FLIP_TOP_BOTTOM = Image.FLIP_TOP_BOTTOM
    ROTATE_90 = Image.ROTATE_90
    ROTATE_180 = Image.ROTATE_180
    ROTATE_270 = Image.ROTATE_270


def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def now_stamp() -> str:
    return datetime.now().strftime('%H%M%S_%d%m')


def is_colab_runtime() -> bool:
    try:
        import google.colab  # type: ignore  # noqa: F401
        return True
    except Exception:
        return False


def auto_num_workers(in_colab: bool) -> int:
    if sys.platform == 'darwin':
        return 0
    cpu_count = max(os.cpu_count() or 2, 2)
    if in_colab:
        return min(4, max(cpu_count - 1, 2))
    return min(2, cpu_count)


def resolve_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'Data').exists() and (candidate / 'runs').exists():
            return candidate
    for candidate in [current, *current.parents]:
        if (candidate / 'Data').exists():
            return candidate
    raise FileNotFoundError(f'Could not locate repo root from {start.resolve()}')


def make_unique_run_root(runs_root: Path, run_slug: str) -> Path:
    safe_slug = ''.join(ch if ch.isalnum() or ch in '_-' else '_' for ch in run_slug).strip('_')
    candidate = runs_root / f'{now_stamp()}_{safe_slug}'
    while candidate.exists():
        time.sleep(1.1)
        candidate = runs_root / f'{now_stamp()}_{safe_slug}'
    return candidate


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=True) + '\n', encoding='utf-8')


## 2. Configuration

In [ ]:
@dataclass
class RunConfig:
    model_id: str = 'fsrcnn_residual_96_40_m8'
    run_slug: str = 'fsrcnn_residual_96_40_m8'
    enable_resume: bool = True
    resume_checkpoint_path: str | None = '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/085328_2604_fsrcnn_residual_96_40_m8/checkpoints/last.pth'
    seed: int = 1337
    train_patch_size: int = 256
    eval_size: int = 256
    batch_size: int = 16
    epochs: int = 2750
    lr: float = 1e-4
    min_lr: float = 1e-6
    weight_decay: float = 1e-4
    warmup_epochs: int = 8
    num_workers: int | None = 2
    prefetch_factor: int = 2
    use_amp: bool = True
    use_ema: bool = True
    ema_decay: float = 0.999
    calibration_count: int = 128

cfg = RunConfig()
in_colab = is_colab_runtime()
if in_colab:
    from google.colab import drive  # type: ignore

    drive.mount('/content/drive', force_remount=False)
    data_root = DRIVE_DATA_ROOT
    runs_root = DRIVE_RUNS_ROOT
    if not data_root.exists():
        raise FileNotFoundError(f'Google Drive data root not found: {data_root}')
else:
    repo_root = resolve_repo_root(Path.cwd())
    data_root = repo_root / 'Data'
    runs_root = repo_root / 'runs'
    if not data_root.exists():
        raise FileNotFoundError(f'Data root not found: {data_root}')

runs_root.mkdir(parents=True, exist_ok=True)
if cfg.num_workers is None:
    cfg.num_workers = auto_num_workers(in_colab)
if not cfg.enable_resume:
    cfg.resume_checkpoint_path = None

run_root = make_unique_run_root(runs_root, cfg.run_slug)
checkpoints_dir = run_root / 'checkpoints'
exports_dir = run_root / 'exports'
preview_dir = exports_dir / 'val_preview'
calibration_dir = exports_dir / 'calibration'
summary_path = run_root / 'summary.json'
metrics_csv_path = run_root / 'metrics.csv'

for directory in [run_root, checkpoints_dir, exports_dir, preview_dir, calibration_dir]:
    directory.mkdir(parents=True, exist_ok=True)

set_global_seed(cfg.seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    torch.backends.cudnn.benchmark = True
    if hasattr(torch.backends.cuda.matmul, 'allow_tf32'):
        torch.backends.cuda.matmul.allow_tf32 = True

print({
    'device': str(device),
    'in_colab': in_colab,
    'data_root': str(data_root),
    'run_root': str(run_root),
    'config': asdict(cfg),
})


## 3. Data

In [ ]:
L3_TRAIN_SPLITS = (
    ('LR_HR/L3_HR_train1', 'L3_LR/L3_L3_train1', 'L3_HR_train1'),
    ('LR_HR/L3_HR_train2', 'L3_LR/L3_L3_train2', 'L3_HR_train2'),
    ('LR_HR/L3_HR_train3', 'L3_LR/L3_LR_train3', 'L3_HR_train3'),
    ('LR_HR/L3_HR_train4', 'L3_LR/L3_LR_train4', 'L3_HR_train4'),
    ('LR_HR/L3_HR_train', 'L3_LR/L3_LR_train', 'L3_HR_train'),
)
L3_VAL_DIRS = ('L3_HR_valid', 'L3_LR_valid')


def scan_image_directory(directory: Path) -> dict[str, Path]:
    if not directory.exists():
        raise FileNotFoundError(f'Missing directory: {directory}')
    return {
        path.stem: path
        for path in directory.iterdir()
        if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
    }


def collect_split_pairs(hr_dir: Path, lr_dir: Path, split_name: str) -> list[tuple[Path, Path, str]]:
    hr_map = scan_image_directory(hr_dir)
    lr_map = scan_image_directory(lr_dir)
    shared = sorted(set(hr_map) & set(lr_map))
    if not shared:
        raise FileNotFoundError(f'No paired files found for {split_name}: HR={hr_dir}, LR={lr_dir}')
    hr_only = sorted(set(hr_map) - set(lr_map))
    lr_only = sorted(set(lr_map) - set(hr_map))
    if hr_only or lr_only:
        print({'split': split_name, 'hr_only_count': len(hr_only), 'lr_only_count': len(lr_only)})
    pairs: list[tuple[Path, Path, str]] = []
    for stem in shared:
        name = stem if split_name == 'val' else f'{split_name}/{stem}'
        pairs.append((lr_map[stem], hr_map[stem], name))
    return pairs


def collect_all_pairs(data_root: Path) -> tuple[list[tuple[Path, Path, str]], list[tuple[Path, Path, str]]]:
    train_pairs: list[tuple[Path, Path, str]] = []
    for hr_rel, lr_rel, split_name in L3_TRAIN_SPLITS:
        hr_dir = data_root / hr_rel
        lr_dir = data_root / lr_rel
        if hr_dir.exists() and lr_dir.exists():
            train_pairs.extend(collect_split_pairs(hr_dir, lr_dir, split_name))
        else:
            print({'skipped_missing_train_split': split_name, 'hr_dir': str(hr_dir), 'lr_dir': str(lr_dir)})
    val_pairs = collect_split_pairs(data_root / L3_VAL_DIRS[0], data_root / L3_VAL_DIRS[1], 'val')
    return train_pairs, val_pairs


class PairedSRDataset(Dataset):
    def __init__(self, pairs: list[tuple[Path, Path, str]], *, train: bool, patch_size: int, eval_size: int, seed: int):
        self.pairs = pairs
        self.train = train
        self.patch_size = patch_size
        self.eval_size = eval_size
        self.seed = seed
        self.epoch = 0

    def set_epoch(self, epoch: int) -> None:
        self.epoch = epoch

    @staticmethod
    def to_tensor(img: Image.Image) -> torch.Tensor:
        arr = np.asarray(img, dtype=np.float32) / 255.0
        return torch.from_numpy(arr).permute(2, 0, 1).contiguous()

    def augment_pair(self, lr: Image.Image, hr: Image.Image, index: int) -> tuple[Image.Image, Image.Image]:
        rng = random.Random((self.seed * 1_000_003) + (self.epoch * 9_973) + index)
        width, height = lr.size
        patch = self.patch_size
        if width < patch or height < patch:
            lr = lr.resize((self.eval_size, self.eval_size), BICUBIC_RESAMPLE)
            hr = hr.resize((self.eval_size, self.eval_size), BICUBIC_RESAMPLE)
            width, height = lr.size
        left = rng.randint(0, width - patch)
        top = rng.randint(0, height - patch)
        box = (left, top, left + patch, top + patch)
        lr = lr.crop(box)
        hr = hr.crop(box)
        if rng.random() < 0.5:
            lr = lr.transpose(FLIP_LEFT_RIGHT)
            hr = hr.transpose(FLIP_LEFT_RIGHT)
        if rng.random() < 0.5:
            lr = lr.transpose(FLIP_TOP_BOTTOM)
            hr = hr.transpose(FLIP_TOP_BOTTOM)
        rot = rng.choice([0, 1, 2, 3])
        if rot:
            rotate_ops = [ROTATE_90, ROTATE_180, ROTATE_270]
            lr = lr.transpose(rotate_ops[rot - 1])
            hr = hr.transpose(rotate_ops[rot - 1])
        return lr, hr

    def __len__(self) -> int:
        return len(self.pairs)

    def __getitem__(self, index: int) -> dict[str, Any]:
        lr_path, hr_path, name = self.pairs[index]
        lr = Image.open(lr_path).convert('RGB')
        hr = Image.open(hr_path).convert('RGB')
        if self.train:
            lr, hr = self.augment_pair(lr, hr, index)
        elif lr.size != (self.eval_size, self.eval_size):
            lr = lr.resize((self.eval_size, self.eval_size), BICUBIC_RESAMPLE)
            hr = hr.resize((self.eval_size, self.eval_size), BICUBIC_RESAMPLE)
        return {'lr': self.to_tensor(lr), 'hr': self.to_tensor(hr), 'name': name}

train_pairs, val_pairs = collect_all_pairs(data_root)
if not train_pairs:
    raise FileNotFoundError(f'No training pairs found under {data_root}')
if not val_pairs:
    raise FileNotFoundError(f'No validation pairs found under {data_root}')

train_ds = PairedSRDataset(train_pairs, train=True, patch_size=cfg.train_patch_size, eval_size=cfg.eval_size, seed=cfg.seed)
val_ds = PairedSRDataset(val_pairs, train=False, patch_size=cfg.train_patch_size, eval_size=cfg.eval_size, seed=cfg.seed)

loader_num_workers = int(cfg.num_workers or 0)
train_loader_kwargs = {'dataset': train_ds, 'batch_size': cfg.batch_size, 'shuffle': True, 'num_workers': loader_num_workers, 'pin_memory': device.type == 'cuda'}
val_loader_kwargs = {'dataset': val_ds, 'batch_size': 1, 'shuffle': False, 'num_workers': loader_num_workers, 'pin_memory': device.type == 'cuda'}
if loader_num_workers > 0:
    train_loader_kwargs['persistent_workers'] = True
    val_loader_kwargs['persistent_workers'] = True
    train_loader_kwargs['prefetch_factor'] = cfg.prefetch_factor
    val_loader_kwargs['prefetch_factor'] = cfg.prefetch_factor

train_loader = DataLoader(**train_loader_kwargs)
val_loader = DataLoader(**val_loader_kwargs)

sample = train_ds[0]
print({
    'data_root': str(data_root),
    'train_pairs': len(train_pairs),
    'val_pairs': len(val_pairs),
    'train_preview': [name for _, _, name in train_pairs[:5]],
    'val_preview': [name for _, _, name in val_pairs[:5]],
    'sample_lr_shape': tuple(sample['lr'].shape),
    'sample_hr_shape': tuple(sample['hr'].shape),
})
assert tuple(sample['lr'].shape) == (3, cfg.eval_size, cfg.eval_size)
assert tuple(sample['hr'].shape) == (3, cfg.eval_size, cfg.eval_size)


## 4. Model

In [ ]:
class FSRCNNResidualNoUpscale(nn.Module):
    def __init__(self, channels: int = 96, shrink_channels: int = 40, mapping_layers: int = 8, slope: float = 0.10):
        super().__init__()
        self.channels = channels
        self.shrink_channels = shrink_channels
        self.mapping_layers = mapping_layers
        self.slope = slope

        # Keep this one Sequential so checkpoint keys stay compatible with earlier runs.
        self.features = nn.Sequential(*self._feature_layers())
        self.tail = nn.Conv2d(channels, 3, 3, padding=1, bias=True)
        self.init_tail_small()

    def _activation(self) -> nn.Module:
        return nn.LeakyReLU(self.slope, inplace=False)

    def _feature_layers(self) -> list[nn.Module]:
        stem = [
            nn.Conv2d(3, self.channels, 5, padding=2, bias=True),
            self._activation(),
        ]
        shrink = [
            nn.Conv2d(self.channels, self.shrink_channels, 1, padding=0, bias=True),
            self._activation(),
        ]
        mapping: list[nn.Module] = []
        for _ in range(self.mapping_layers):
            mapping.extend([
                nn.Conv2d(self.shrink_channels, self.shrink_channels, 3, padding=1, bias=True),
                self._activation(),
            ])
        expand = [
            nn.Conv2d(self.shrink_channels, self.channels, 1, padding=0, bias=True),
            self._activation(),
        ]
        return [*stem, *shrink, *mapping, *expand]

    def architecture_summary(self) -> dict[str, str | int]:
        return {
            'stem': f'Conv5x5 3->{self.channels} + LeakyReLU',
            'shrink': f'Conv1x1 {self.channels}->{self.shrink_channels} + LeakyReLU',
            'mapping': f'{self.mapping_layers}x Conv3x3 {self.shrink_channels}->{self.shrink_channels} + LeakyReLU',
            'expand': f'Conv1x1 {self.shrink_channels}->{self.channels} + LeakyReLU',
            'tail': f'Conv3x3 {self.channels}->3 predicts residual delta',
            'forward': 'output = input + tail(features(input))',
        }

    def init_tail_small(self) -> None:
        nn.init.kaiming_normal_(self.tail.weight, nonlinearity='linear')
        self.tail.weight.data.mul_(1e-3)
        if self.tail.bias is not None:
            nn.init.zeros_(self.tail.bias)

    def predict_delta(self, x: torch.Tensor) -> torch.Tensor:
        return self.tail(self.features(x))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.predict_delta(x)


def count_parameters(module: nn.Module) -> int:
    return int(sum(param.numel() for param in module.parameters()))


def operator_audit(module: nn.Module) -> dict[str, int]:
    counts: dict[str, int] = {}
    for child in module.modules():
        if len(list(child.children())) != 0:
            continue
        name = child.__class__.__name__
        counts[name] = counts.get(name, 0) + 1
    return dict(sorted(counts.items()))

model = FSRCNNResidualNoUpscale().to(device)
eval_model = FSRCNNResidualNoUpscale().to(device)
param_count = count_parameters(model)
op_counts = operator_audit(model)
dummy = torch.zeros(1, 3, cfg.eval_size, cfg.eval_size, device=device)
with torch.no_grad():
    dummy_out = model(dummy)

print({
    'model_id': cfg.model_id,
    'params': param_count,
    'operator_audit': op_counts,
    'input_shape': tuple(dummy.shape),
    'output_shape': tuple(dummy_out.shape),
    'architecture': model.architecture_summary(),
})
assert param_count == 133_227, param_count
assert tuple(dummy_out.shape) == (1, 3, cfg.eval_size, cfg.eval_size)
assert set(op_counts).issubset({'Conv2d', 'LeakyReLU'}), op_counts


## 5. Training And Validation

Train with AMP, EMA, AdamW, cosine learning rate decay, PSNR validation, checkpointing, and validation previews.

In [ ]:
class EMA:
    def __init__(self, model: nn.Module, decay: float):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model: nn.Module) -> None:
        for key, value in model.state_dict().items():
            self.shadow[key].mul_(self.decay).add_(value.detach(), alpha=1.0 - self.decay)

    @torch.no_grad()
    def copy_to(self, model: nn.Module) -> None:
        model.load_state_dict(self.shadow, strict=True)


def make_grad_scaler(enabled: bool):
    if hasattr(torch, 'amp') and hasattr(torch.amp, 'GradScaler'):
        return torch.amp.GradScaler('cuda', enabled=enabled)
    return torch.cuda.amp.GradScaler(enabled=enabled)


def autocast_context(device_type: str, enabled: bool):
    if not enabled:
        return nullcontext()
    if hasattr(torch, 'amp') and hasattr(torch.amp, 'autocast'):
        return torch.amp.autocast(device_type=device_type, enabled=enabled)
    return torch.cuda.amp.autocast(enabled=enabled)


@torch.no_grad()
def batch_psnr(pred: torch.Tensor, target: torch.Tensor) -> float:
    mse = F.mse_loss(pred, target, reduction='none').mean(dim=(1, 2, 3))
    psnr = 10.0 * torch.log10(1.0 / torch.clamp(mse, min=1e-12))
    return float(psnr.mean().item())


def compute_loss(pred: torch.Tensor, lr: torch.Tensor, hr: torch.Tensor) -> torch.Tensor:
    mse = F.mse_loss(pred, hr)
    residual_l1 = F.l1_loss(pred - lr, hr - lr)
    return 0.8 * mse + 0.2 * residual_l1


def lr_at_epoch(epoch_idx: int, *, base_lr: float, min_lr: float, warmup_epochs: int, total_epochs: int) -> float:
    step = epoch_idx + 1
    if warmup_epochs > 0 and step <= warmup_epochs:
        return base_lr * step / max(1, warmup_epochs)
    progress = max(0.0, (step - warmup_epochs) / max(1, total_epochs - warmup_epochs))
    cosine = 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))
    return min_lr + cosine * (base_lr - min_lr)


@torch.inference_mode()
def evaluate_metrics(model: nn.Module, loader: DataLoader, device: torch.device) -> dict[str, float]:
    model.eval()
    pred_meter = 0.0
    input_meter = 0.0
    count = 0
    for batch in loader:
        lr = batch['lr'].to(device, non_blocking=(device.type == 'cuda'))
        hr = batch['hr'].to(device, non_blocking=(device.type == 'cuda'))
        pred = model(lr).clamp(0.0, 1.0)
        pred_meter += batch_psnr(pred, hr)
        input_meter += batch_psnr(lr, hr)
        count += 1
    val_psnr = pred_meter / max(count, 1)
    input_psnr = input_meter / max(count, 1)
    return {'val_psnr': float(val_psnr), 'input_psnr': float(input_psnr), 'delta_psnr': float(val_psnr - input_psnr)}


def tensor_to_uint8_image(x: torch.Tensor) -> Image.Image:
    x = x.detach().cpu().clamp(0.0, 1.0)
    arr = (x.permute(1, 2, 0).numpy() * 255.0).round().astype(np.uint8)
    return Image.fromarray(arr)


@torch.no_grad()
def write_val_preview(model: nn.Module, loader: DataLoader, out_dir: Path, device: torch.device) -> dict[str, str]:
    out_dir.mkdir(parents=True, exist_ok=True)
    batch = next(iter(loader))
    lr = batch['lr'].to(device, non_blocking=(device.type == 'cuda'))
    hr = batch['hr'].to(device, non_blocking=(device.type == 'cuda'))
    pred = model(lr).clamp(0.0, 1.0)
    name = batch['name'][0]
    lr_path = out_dir / 'val_preview_input.png'
    pred_path = out_dir / 'val_preview_pred.png'
    hr_path = out_dir / 'val_preview_target.png'
    tensor_to_uint8_image(lr[0]).save(lr_path)
    tensor_to_uint8_image(pred[0]).save(pred_path)
    tensor_to_uint8_image(hr[0]).save(hr_path)
    return {'sample_name': str(name), 'input': str(lr_path), 'prediction': str(pred_path), 'target': str(hr_path)}


def checkpoint_payload(model: nn.Module, ema: EMA | None, optimizer: torch.optim.Optimizer, epoch: int, best_val_psnr: float, row: dict[str, Any]) -> dict[str, Any]:
    return {
        'config': asdict(cfg),
        'epoch': int(epoch),
        'model_state_dict': model.state_dict(),
        'ema_state_dict': ema.shadow if ema is not None else None,
        'optimizer_state_dict': optimizer.state_dict(),
        'best_val_psnr': float(best_val_psnr),
        'latest_metrics': row,
        'model_params': param_count,
        'operator_audit': op_counts,
    }


def load_checkpoint_for_eval(checkpoint_path: Path, device: torch.device) -> nn.Module:
    checkpoint = torch.load(checkpoint_path, map_location=device)
    loaded = FSRCNNResidualNoUpscale().to(device)
    state = checkpoint.get('ema_state_dict') or checkpoint['model_state_dict']
    loaded.load_state_dict(state, strict=True)
    loaded.eval()
    return loaded

optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scaler = make_grad_scaler(cfg.use_amp and device.type == 'cuda')
ema = EMA(model, cfg.ema_decay) if cfg.use_ema else None
best_val_psnr = float('-inf')
start_epoch = 0

metrics_header = ['epoch', 'epoch_time_sec', 'lr', 'train_loss', 'train_psnr', 'val_psnr', 'input_psnr', 'delta_psnr', 'best_val_psnr']
best_ckpt_path = checkpoints_dir / 'best.pth'
last_ckpt_path = checkpoints_dir / 'last.pth'

if cfg.enable_resume:
    if cfg.resume_checkpoint_path is None:
        raise ValueError('cfg.enable_resume=True requires cfg.resume_checkpoint_path')
    resume_checkpoint_path = Path(cfg.resume_checkpoint_path)
    if not resume_checkpoint_path.exists():
        raise FileNotFoundError(f'Resume checkpoint not found: {resume_checkpoint_path}')
    resume_checkpoint = torch.load(resume_checkpoint_path, map_location=device)
    model.load_state_dict(resume_checkpoint['model_state_dict'], strict=True)
    if 'optimizer_state_dict' in resume_checkpoint:
        optimizer.load_state_dict(resume_checkpoint['optimizer_state_dict'])
    if ema is not None and resume_checkpoint.get('ema_state_dict') is not None:
        ema.shadow = {key: value.detach().clone().to(device) for key, value in resume_checkpoint['ema_state_dict'].items()}
    best_val_psnr = float(resume_checkpoint.get('best_val_psnr', best_val_psnr))
    start_epoch = int(resume_checkpoint.get('epoch', 0))
    print({
        'resume_checkpoint_path': str(resume_checkpoint_path),
        'resumed_from_epoch': start_epoch,
        'target_total_epochs': cfg.epochs,
        'best_val_psnr': best_val_psnr,
    })

if start_epoch >= cfg.epochs:
    raise ValueError(f'Resume checkpoint epoch {start_epoch} is already >= target epochs {cfg.epochs}')

with metrics_csv_path.open('w', newline='', encoding='utf-8') as f:
    csv.DictWriter(f, fieldnames=metrics_header).writeheader()

for epoch in range(start_epoch, cfg.epochs):
    train_ds.set_epoch(epoch)
    epoch_lr = lr_at_epoch(epoch, base_lr=cfg.lr, min_lr=cfg.min_lr, warmup_epochs=cfg.warmup_epochs, total_epochs=cfg.epochs)
    for group in optimizer.param_groups:
        group['lr'] = epoch_lr

    epoch_start = time.time()
    model.train()
    train_loss_sum = 0.0
    train_psnr_sum = 0.0
    train_batches = 0

    for batch in train_loader:
        lr = batch['lr'].to(device, non_blocking=(device.type == 'cuda'))
        hr = batch['hr'].to(device, non_blocking=(device.type == 'cuda'))
        optimizer.zero_grad(set_to_none=True)
        with autocast_context(device.type, enabled=(cfg.use_amp and device.type == 'cuda')):
            pred = model(lr)
            loss = compute_loss(pred, lr, hr)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        if ema is not None:
            ema.update(model)
        train_loss_sum += float(loss.item())
        train_psnr_sum += batch_psnr(pred.detach().clamp(0.0, 1.0), hr)
        train_batches += 1

    if ema is not None:
        ema.copy_to(eval_model)
    else:
        eval_model.load_state_dict(model.state_dict())
    eval_model.eval()

    avg_train_loss = train_loss_sum / max(train_batches, 1)
    avg_train_psnr = train_psnr_sum / max(train_batches, 1)
    val_metrics = evaluate_metrics(eval_model, val_loader, device)
    is_best = val_metrics['val_psnr'] > best_val_psnr
    if is_best:
        best_val_psnr = val_metrics['val_psnr']
    row = {
        'epoch': epoch + 1,
        'epoch_time_sec': round(time.time() - epoch_start, 2),
        'lr': epoch_lr,
        'train_loss': round(avg_train_loss, 6),
        'train_psnr': round(avg_train_psnr, 4),
        'val_psnr': round(val_metrics['val_psnr'], 4),
        'input_psnr': round(val_metrics['input_psnr'], 4),
        'delta_psnr': round(val_metrics['delta_psnr'], 4),
        'best_val_psnr': round(best_val_psnr, 4),
    }
    print(row)
    with metrics_csv_path.open('a', newline='', encoding='utf-8') as f:
        csv.DictWriter(f, fieldnames=metrics_header).writerow(row)
    payload = checkpoint_payload(model, ema, optimizer, epoch + 1, best_val_psnr, row)
    torch.save(payload, last_ckpt_path)
    if is_best:
        torch.save(payload, best_ckpt_path)


## 6. Validation Summary

Load the best checkpoint, compute validation PSNR against the raw LR baseline, and write preview images.

In [ ]:
export_model = load_checkpoint_for_eval(best_ckpt_path, device)
validation_summary = evaluate_metrics(export_model, val_loader, device)
preview_info = write_val_preview(export_model, val_loader, preview_dir, device)
print({'validation_summary': validation_summary, 'best_checkpoint_path': str(best_ckpt_path), 'last_checkpoint_path': str(last_ckpt_path), 'preview': preview_info})


## 7. Export And Submission Artifacts

Export ONNX, run ONNX checker and ONNX Runtime parity, export training-derived calibration images, generate an MXQ handoff script, and write a final summary.

In [ ]:
def export_calibration_images(pairs: list[tuple[Path, Path, str]], out_dir: Path, count: int, eval_size: int) -> dict[str, Any]:
    out_dir.mkdir(parents=True, exist_ok=True)
    selected = pairs[: min(count, len(pairs))]
    manifest_items: list[dict[str, Any]] = []
    for index, (lr_path, _, name) in enumerate(selected):
        image = Image.open(lr_path).convert('RGB')
        if image.size != (eval_size, eval_size):
            image = image.resize((eval_size, eval_size), BICUBIC_RESAMPLE)
        image_path = out_dir / f'{index:03d}_{Path(name).name}.png'
        image.save(image_path)
        manifest_items.append({'index': index, 'name': name, 'source_lr': str(lr_path), 'image_path': str(image_path), 'derived_from_training': True})
    manifest = {'count': len(manifest_items), 'source': 'training_pairs', 'eval_size': eval_size, 'derived_from_training': True, 'items': manifest_items}
    manifest_path = out_dir / 'manifest.json'
    write_json(manifest_path, manifest)
    return {'calibration_dir': str(out_dir), 'manifest_path': str(manifest_path), 'count': len(manifest_items), 'derived_from_training': True}

conversion_script = """#!/usr/bin/env python3
from __future__ import annotations

import argparse
import json
import shlex
import subprocess
from pathlib import Path


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description='Lab 3 ONNX-to-MXQ handoff script for FSRCNN residual.')
    parser.add_argument('--onnx', type=Path, required=True)
    parser.add_argument('--calibration-dir', type=Path, required=True)
    parser.add_argument('--output', type=Path, required=True)
    parser.add_argument('--command-template', default='', help='Optional command with {onnx}, {calibration}, {output} placeholders.')
    parser.add_argument('--dry-run', action='store_true')
    return parser.parse_args()


def main() -> None:
    args = parse_args()
    args.onnx = args.onnx.resolve()
    args.calibration_dir = args.calibration_dir.resolve()
    args.output = args.output.resolve()
    if not args.onnx.exists():
        raise FileNotFoundError(f'ONNX file not found: {args.onnx}')
    if not args.calibration_dir.exists():
        raise FileNotFoundError(f'Calibration directory not found: {args.calibration_dir}')
    payload = {
        'onnx': str(args.onnx),
        'calibration_dir': str(args.calibration_dir),
        'output': str(args.output),
        'status': 'handoff_only',
        'command_template': args.command_template,
    }
    if args.command_template:
        command = args.command_template.format(
            onnx=shlex.quote(str(args.onnx)),
            calibration=shlex.quote(str(args.calibration_dir)),
            output=shlex.quote(str(args.output)),
        )
        payload['command'] = command
        if not args.dry_run:
            completed = subprocess.run(command, shell=True, check=False, capture_output=True, text=True)
            payload.update({'status': 'completed' if completed.returncode == 0 else 'failed', 'returncode': completed.returncode, 'stdout': completed.stdout, 'stderr': completed.stderr})
    args.output.parent.mkdir(parents=True, exist_ok=True)
    handoff_path = args.output.with_suffix('.handoff.json')
    handoff_path.write_text(json.dumps(payload, indent=2, sort_keys=True) + '\n', encoding='utf-8')
    print(json.dumps(payload, indent=2, sort_keys=True))


if __name__ == '__main__':
    main()
"""

onnx_path = exports_dir / f'{cfg.model_id}.onnx'
mxq_target_path = exports_dir / f'{cfg.model_id}.mxq'
conversion_script_path = exports_dir / 'convert_fsrcnn_residual_mxq.py'
conversion_script_path.write_text(conversion_script, encoding='utf-8')

cpu_export_model = load_checkpoint_for_eval(best_ckpt_path, torch.device('cpu')).cpu().eval()
dummy = torch.randn(1, 3, cfg.eval_size, cfg.eval_size)
torch.onnx.export(cpu_export_model, dummy, str(onnx_path), input_names=['input'], output_names=['output'], opset_version=17, dynamo=False)

onnx_model = onnx.load(str(onnx_path))
onnx.checker.check_model(onnx_model)
node_ops = [node.op_type for node in onnx_model.graph.node]
op_histogram = {op: node_ops.count(op) for op in sorted(set(node_ops))}
onnx_summary = {'onnx_path': str(onnx_path), 'checked': True, 'graph_op_count': len(node_ops), 'op_histogram': op_histogram}
print(onnx_summary)

parity_result = {'attempted': False, 'onnxruntime_available': False, 'max_abs_diff': None, 'mean_abs_diff': None}
try:
    import onnxruntime as ort
    parity_result['attempted'] = True
    parity_result['onnxruntime_available'] = True
    parity_input = torch.randn(1, 3, cfg.eval_size, cfg.eval_size)
    with torch.no_grad():
        torch_out = cpu_export_model(parity_input).detach().cpu().numpy()
    ort_session = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
    ort_out = ort_session.run(['output'], {'input': parity_input.numpy()})[0]
    diff = np.abs(torch_out - ort_out)
    parity_result['max_abs_diff'] = float(diff.max())
    parity_result['mean_abs_diff'] = float(diff.mean())
except ImportError:
    print('onnxruntime not installed; skipping CPU parity check')
print(parity_result)

calibration_summary = export_calibration_images(train_pairs, calibration_dir, cfg.calibration_count, cfg.eval_size)
mxq_command = f'python {conversion_script_path} --onnx {onnx_path} --calibration-dir {calibration_dir} --output {mxq_target_path} --dry-run'
mxq_handoff = {
    'conversion_script_path': str(conversion_script_path),
    'onnx_input_path': str(onnx_path),
    'calibration_manifest_path': calibration_summary['manifest_path'],
    'mxq_target_path': str(mxq_target_path),
    'command': mxq_command,
    'requested_compile': False,
}
summary = {
    'model_id': cfg.model_id,
    'run_root': str(run_root),
    'config': asdict(cfg),
    'data': {'data_root': str(data_root), 'train_pairs': len(train_pairs), 'val_pairs': len(val_pairs), 'train_splits': L3_TRAIN_SPLITS, 'val_dirs': L3_VAL_DIRS},
    'model': {'params': param_count, 'operator_audit': op_counts, 'input_shape': [1, 3, cfg.eval_size, cfg.eval_size], 'output_shape': [1, 3, cfg.eval_size, cfg.eval_size]},
    'validation': validation_summary,
    'preview': preview_info,
    'artifacts': {
        'best_checkpoint': str(best_ckpt_path),
        'last_checkpoint': str(last_ckpt_path),
        'onnx': str(onnx_path),
        'calibration_manifest': calibration_summary['manifest_path'],
        'conversion_script': str(conversion_script_path),
        'mxq_target': str(mxq_target_path),
        'metrics_csv': str(metrics_csv_path),
    },
    'onnx': onnx_summary,
    'onnx_parity': parity_result,
    'calibration': calibration_summary,
    'mxq_handoff': mxq_handoff,
}
write_json(summary_path, summary)
print({'summary_path': str(summary_path), **summary['artifacts']})
